In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import request

In [2]:
# dataset: https://zenodo.org/records/17266923
# dataset repo: https://github.com/vintagedon/steam-dataset-2025/tree/main

In [ ]:
def get_steam_game_details(app_id):
    url = "https://store.steampowered.com/api/appdetails"
    params = {
        "appids": app_id
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        raise Exception(f"Request failed with status code {response.status_code}")

    data = response.json()

    # return data

    # Steam wraps the response under the app ID key
    if not data[str(app_id)]["success"]:
        raise Exception("Failed to fetch game data")

    game_data = data[str(app_id)]["data"]

    return {
        "name": game_data.get("name"),
        "type": game_data.get("type"),
        "release_date": game_data.get("release_date", {}).get("date"),
        "price": game_data.get("price_overview", {}).get("final_formatted"),
        "genres": [g["description"] for g in game_data.get("genres", [])],
        "description": game_data.get("short_description"),
        "pc_requirements": game_data.get("pc_requirements"),
        "mac_requirements": game_data.get("mac_requirements"),
        "linux_requirements": game_data.get("linux_requirements"),
    }

In [ ]:
get_steam_game_details(570) #returns a dictionary with strings straight from JSON

In [ ]:
from bs4 import BeautifulSoup

def parse_requirements_html(html_string):
    """Parse a single HTML requirements block into a dict."""
    if not html_string:
        return {}

    soup = BeautifulSoup(html_string, "html.parser")
    result = {}

    for li in soup.find_all("li"):
        strong = li.find("strong")
        if not strong:
            continue

        key = strong.get_text(strip=True).replace(":", "").replace("*", "").strip()
        strong.extract()  # remove label
        value = li.get_text(" ", strip=True)

        result[key] = value

    return result


def extract_system_requirements(game_data):
    """
    Takes the 'data' field from Steam API response and parses
    pc/mac/linux requirements into structured dicts.
    """
    systems = ["pc_requirements", "mac_requirements", "linux_requirements"]
    parsed = {}

    for system in systems:
        system_data = game_data.get(system)
        if not system_data:
            continue

        parsed[system] = {}

        for level in ["minimum", "recommended"]:
            html = system_data.get(level)
            if html:
                parsed[system][level] = parse_requirements_html(html)

    return parsed


In [ ]:
# --- Example usage with real API response ---
def get_parsed_requirements(api_response, app_id): #useless function for now, maybe if we collect a bunch of API responses and put them in
    game_data = api_response[str(app_id)]["data"]
    return extract_system_requirements(game_data)

In [ ]:
# Example usage

parsed = extract_system_requirements(details)
print(parsed)
# import pprint
# pprint.pprint(parsed)